# 05 — Analisis dengan Spark SQL (Bab 2.4a)Notebook ini mengimplementasikan analisis eksploratif menggunakan **Spark SQL** terhadap data transaksi ritel.

## 5.1 Inisialisasi & Load dari Parquet

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("05_SparkSQL_Analysis") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

# Load dari Parquet (lebih cepat dari CSV)
df = spark.read.parquet("/output/retail_parquet")
df.createOrReplaceTempView("transactions")
print(f"✅ Loaded {df.count()} rows from Parquet")

## 5.2 Query 1: Revenue per Kategori Produk

In [ ]:
print("=== REVENUE PER KATEGORI PRODUK ===")
spark.sql("""
    SELECT 
        Product_Category,
        COUNT(*) as total_transaksi,
        SUM(Total_Amount) as total_revenue,
        ROUND(AVG(Total_Amount), 1) as avg_per_transaksi,
        SUM(Quantity) as total_unit_terjual
    FROM transactions
    GROUP BY Product_Category
    ORDER BY total_revenue DESC
""").show()

## 5.3 Query 2: Distribusi Gender

In [ ]:
print("=== DISTRIBUSI GENDER ===")
spark.sql("""
    SELECT 
        Gender,
        COUNT(*) as jumlah_pelanggan,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 1) as persentase,
        SUM(Total_Amount) as total_spending,
        ROUND(AVG(Total_Amount), 1) as avg_spending
    FROM transactions
    GROUP BY Gender
""").show()

## 5.4 Query 3: Analisis per Kelompok Usia

In [ ]:
print("=== SPENDING PER KELOMPOK USIA ===")
spark.sql("""
    SELECT 
        CASE 
            WHEN Age BETWEEN 18 AND 25 THEN '18-25 (Muda)'
            WHEN Age BETWEEN 26 AND 35 THEN '26-35 (Dewasa Muda)'
            WHEN Age BETWEEN 36 AND 45 THEN '36-45 (Dewasa)'
            WHEN Age BETWEEN 46 AND 55 THEN '46-55 (Dewasa Atas)'
            ELSE '56-64 (Pra-Lansia)'
        END as kelompok_usia,
        COUNT(*) as jumlah,
        SUM(Total_Amount) as total_spending,
        ROUND(AVG(Total_Amount), 1) as avg_spending,
        ROUND(AVG(Quantity), 1) as avg_quantity
    FROM transactions
    GROUP BY kelompok_usia
    ORDER BY total_spending DESC
""").show()

## 5.5 Query 4: Tren Penjualan Bulanan

In [ ]:
print("=== TREN PENJUALAN BULANAN (2023) ===")
spark.sql("""
    SELECT 
        MONTH(Date) as bulan,
        COUNT(*) as total_transaksi,
        SUM(Total_Amount) as total_revenue,
        ROUND(AVG(Total_Amount), 1) as avg_per_transaksi
    FROM transactions
    WHERE YEAR(Date) = 2023
    GROUP BY MONTH(Date)
    ORDER BY bulan
""").show(12)

## 5.6 Query 5: Top Spender per Kategori

In [ ]:
print("=== TOP 5 PELANGGAN PER KATEGORI ===")
spark.sql("""
    SELECT * FROM (
        SELECT 
            Product_Category,
            Customer_ID,
            Total_Amount,
            ROW_NUMBER() OVER (PARTITION BY Product_Category ORDER BY Total_Amount DESC) as rank
        FROM transactions
    )
    WHERE rank <= 5
    ORDER BY Product_Category, rank
""").show(15)

## 5.7 Query 6: Distribusi Harga

In [ ]:
print("=== DISTRIBUSI PRICE TIER ===")
spark.sql("""
    SELECT 
        Price_per_Unit as harga,
        CASE
            WHEN Price_per_Unit IN (25, 30) THEN 'Budget'
            WHEN Price_per_Unit = 50 THEN 'Mid-Range'
            ELSE 'Premium'
        END as price_tier,
        COUNT(*) as jumlah_transaksi,
        SUM(Total_Amount) as total_revenue
    FROM transactions
    GROUP BY Price_per_Unit, price_tier
    ORDER BY Price_per_Unit
""").show()

## 5.8 Query 7: Cross-Tabulation Gender × Kategori

In [ ]:
print("=== CROSS-TAB: GENDER × KATEGORI ===")
spark.sql("""
    SELECT 
        Gender,
        SUM(CASE WHEN Product_Category = 'Beauty' THEN Total_Amount ELSE 0 END) as Beauty,
        SUM(CASE WHEN Product_Category = 'Clothing' THEN Total_Amount ELSE 0 END) as Clothing,
        SUM(CASE WHEN Product_Category = 'Electronics' THEN Total_Amount ELSE 0 END) as Electronics,
        SUM(Total_Amount) as Grand_Total
    FROM transactions
    GROUP BY Gender
""").show()